In [ ]:
import os
import json
import copy
import random
import warnings
from typing import List, Dict, Tuple, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset

warnings.filterwarnings("ignore", message=".*VisibleDeprecationWarning.*")
warnings.filterwarnings("ignore", message=".*dtype\\(\\): align should be passed.*")

QUICK_RUN = False
SAVE_PATH = "./outputs/lwf_10task.json"
REPLAY_BUFFER_CAPACITY = 2000
N_TASKS = 10
CLASSES_PER_TASK = 10
TOTAL_CLASSES = N_TASKS * CLASSES_PER_TASK
EPOCHS_BASE = 200 if not QUICK_RUN else 5
EPOCHS_FUSION = 80 if not QUICK_RUN else 5
BATCH_SIZE = 128
USE_AMP = True
FORCE_CPU = False
NUM_WORKERS = 2
TEMPERATURE = 2.0  # Temperature for distillation

def set_deterministic(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def get_device(force_cpu: bool = False) -> torch.device:
    if force_cpu or not torch.cuda.is_available():
        return torch.device("cpu")
    return torch.device("cuda")

def save_progress(obj: Dict, path: str = SAVE_PATH) -> None:
    directory = os.path.dirname(path)
    if directory:
        os.makedirs(directory, exist_ok=True)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)

class CIFAR100TaskDataset(Dataset):
    def __init__(self, dataset: Dataset, classes: List[int]):
        self.dataset = dataset
        self.classes = classes
        self.indices = [i for i, label in enumerate(dataset.targets) if label in classes]

    def __len__(self) -> int:
        return len(self.indices)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, int]:
        img, label = self.dataset[self.indices[idx]]
        return img, label

class TensorDatasetWrapper(Dataset):
    def __init__(self, images: torch.Tensor, labels: torch.Tensor):
        self.images = images
        self.labels = labels

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, int]:
        return self.images[idx], self.labels[idx]

def get_task_classes(task_id: int) -> List[int]:
    start_class = (task_id - 1) * CLASSES_PER_TASK
    end_class = task_id * CLASSES_PER_TASK
    return list(range(start_class, end_class))

def get_task_loaders(
    train_ds: Dataset,
    test_ds: Dataset,
    task_id: int,
    batch_size: int = BATCH_SIZE,
    num_workers: int = NUM_WORKERS,
) -> Tuple[DataLoader, DataLoader]:
    task_classes = get_task_classes(task_id)
    train_subset = CIFAR100TaskDataset(train_ds, task_classes)
    test_subset = CIFAR100TaskDataset(test_ds, task_classes)
    pin_memory = torch.cuda.is_available() and not FORCE_CPU

    train_loader = DataLoader(
        train_subset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=pin_memory,
    )
    test_loader = DataLoader(
        test_subset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=pin_memory,
    )
    return train_loader, test_loader

class DynamicReplayBuffer:
    def __init__(self, capacity_per_class: int = REPLAY_BUFFER_CAPACITY):
        self.capacity_per_class = capacity_per_class
        self.storage: Dict[int, List[Tuple[torch.Tensor, float]]] = {}

    def add_exemplars_from_dataset(
        self,
        dataset: Dataset,
        classes: List[int],
        feature_extractor: nn.Module,
        device: torch.device,
        batch_size: int = BATCH_SIZE,
        num_workers: int = NUM_WORKERS,
    ) -> None:
        pin_memory = torch.cuda.is_available() and not FORCE_CPU
        loader = DataLoader(
            CIFAR100TaskDataset(dataset, classes),
            batch_size=batch_size,
            shuffle=False,
            num_workers=num_workers,
            pin_memory=pin_memory,
        )

        by_class_images: Dict[int, List[torch.Tensor]] = {c: [] for c in classes}
        by_class_feats: Dict[int, List[torch.Tensor]] = {c: [] for c in classes}

        feature_extractor.eval()
        with torch.no_grad():
            for x, y in loader:
                x = x.to(device)
                feats = feature_extractor(x)
                feats = F.normalize(F.adaptive_avg_pool2d(feats, 1).flatten(1), p=2, dim=1).cpu()
                for img, label, feat in zip(x.cpu(), y.cpu(), feats):
                    label_i = int(label.item())
                    by_class_images[label_i].append(img)
                    by_class_feats[label_i].append(feat)

        for cls in classes:
            if len(by_class_images[cls]) == 0:
                continue
            feats = torch.stack(by_class_feats[cls])
            centroid = F.normalize(feats.mean(dim=0, keepdim=True), p=2, dim=1)
            scores = torch.sum(feats * centroid, dim=1)
            k = min(self.capacity_per_class, scores.numel())
            topk = torch.topk(scores, k=k, largest=True).indices.tolist()
            self.storage[cls] = [(by_class_images[cls][idx], float(scores[idx].item())) for idx in topk]

    def get_loader(
        self,
        batch_size: int = BATCH_SIZE,
        num_workers: int = NUM_WORKERS,
        shuffle: bool = True,
    ) -> Optional[DataLoader]:
        if not self.storage:
            return None

        images = []
        labels = []
        for cls in sorted(self.storage.keys()):
            for img, _ in self.storage[cls]:
                images.append(img)
                labels.append(cls)

        if not images:
            return None

        images_t = torch.stack(images)
        labels_t = torch.tensor(labels, dtype=torch.long)
        pin_memory = torch.cuda.is_available() and not FORCE_CPU
        return DataLoader(
            TensorDatasetWrapper(images_t, labels_t),
            batch_size=batch_size,
            shuffle=shuffle,
            num_workers=num_workers,
            pin_memory=pin_memory,
        )

    def __len__(self) -> int:
        return sum(len(v) for v in self.storage.values())

class ResNet18(nn.Module):
    def __init__(self, num_classes: int = 10):
        super().__init__()
        resnet = torchvision.models.resnet18(weights=None)
        resnet.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        resnet.maxpool = nn.Identity()
        self.features = nn.Sequential(*list(resnet.children())[:-1])
        self.classifier = nn.Linear(512, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        features = self.features(x)
        features = F.adaptive_avg_pool2d(features, 1).flatten(1)
        return self.classifier(features)

    def extract_features(self, x: torch.Tensor) -> torch.Tensor:
        return F.normalize(F.adaptive_avg_pool2d(self.features(x), 1).flatten(1), p=2, dim=1)

def evaluate(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
) -> float:
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            pred = logits.argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.size(0)
    return 100.0 * correct / total if total > 0 else 0.0

def train_base_model(
    model: nn.Module,
    train_loader: DataLoader,
    device: torch.device,
    amp_ok: bool,
) -> None:
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS_BASE)
    scaler = torch.amp.GradScaler("cuda") if amp_ok else None

    for _ in range(EPOCHS_BASE):
        model.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            if amp_ok:
                with torch.amp.autocast("cuda"):
                    logits = model(x)
                    loss = F.cross_entropy(logits, y)
                scaler.scale(loss).backward()
                scaler.step(opt)
                scaler.update()
            else:
                logits = model(x)
                loss = F.cross_entropy(logits, y)
                loss.backward()
                opt.step()
        sched.step()

def train_incremental_model(
    model: nn.Module,
    old_model: nn.Module,
    train_loader: DataLoader,
    replay_loader: DataLoader,
    device: torch.device,
    amp_ok: bool,
) -> None:
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS_FUSION)
    scaler = torch.amp.GradScaler("cuda") if amp_ok else None

    for _ in range(EPOCHS_FUSION):
        model.train()
        replay_iter = iter(replay_loader)
        for x_curr, y_curr in train_loader:
            x_curr, y_curr = x_curr.to(device), y_curr.to(device)
            try:
                x_replay, y_replay = next(replay_iter)
            except StopIteration:
                replay_iter = iter(replay_loader)
                x_replay, y_replay = next(replay_iter)
            x_replay, y_replay = x_replay.to(device), y_replay.to(device)

            opt.zero_grad()
            if amp_ok:
                with torch.amp.autocast("cuda"):
                    logits_curr = model(x_curr)
                    logits_replay = model(x_replay)

                    # Knowledge Distillation Loss
                    with torch.no_grad():
                        old_logits = old_model(x_replay)

                    # 🔴 CRITICAL FIX: Slice the student logits to match the old teacher's classes
                    old_num_classes = old_logits.size(1)
                    student_logits_for_kd = logits_replay[:, :old_num_classes]

                    loss_kd = F.kl_div(
                        F.log_softmax(student_logits_for_kd / TEMPERATURE, dim=1),
                        F.softmax(old_logits / TEMPERATURE, dim=1),
                        reduction="batchmean",
                    ) * (TEMPERATURE ** 2)

                    loss_curr = F.cross_entropy(logits_curr, y_curr)
                    loss_replay = F.cross_entropy(logits_replay, y_replay)
                    loss = loss_curr + loss_replay + loss_kd
                scaler.scale(loss).backward()
                scaler.step(opt)
                scaler.update()
            else:
                logits_curr = model(x_curr)
                logits_replay = model(x_replay)

                # Knowledge Distillation Loss
                with torch.no_grad():
                    old_logits = old_model(x_replay)

                # 🔴 CRITICAL FIX: Slice the student logits to match the old teacher's classes
                old_num_classes = old_logits.size(1)
                student_logits_for_kd = logits_replay[:, :old_num_classes]

                loss_kd = F.kl_div(
                    F.log_softmax(student_logits_for_kd / TEMPERATURE, dim=1),
                    F.softmax(old_logits / TEMPERATURE, dim=1),
                    reduction="batchmean",
                ) * (TEMPERATURE ** 2)

                loss_curr = F.cross_entropy(logits_curr, y_curr)
                loss_replay = F.cross_entropy(logits_replay, y_replay)
                loss = loss_curr + loss_replay + loss_kd
                loss.backward()
                opt.step()
        sched.step()

def expand_classifier(model: nn.Module, new_num_classes: int) -> None:
    old_classifier = model.classifier
    new_classifier = nn.Linear(512, new_num_classes).to(old_classifier.weight.device)
    with torch.no_grad():
        new_classifier.weight[:old_classifier.out_features] = old_classifier.weight
        new_classifier.bias[:old_classifier.out_features] = old_classifier.bias
    model.classifier = new_classifier

def train_lwf_10task(
    seed: int = 0,
    quick_run: bool = QUICK_RUN,
    save_path: str = SAVE_PATH,
) -> Dict:
    set_deterministic(seed)
    device = get_device(FORCE_CPU)
    amp_ok = USE_AMP and device.type == "cuda"

    print(f"🚀 Training LwF on Split CIFAR-100 | Seed: {seed} | Device: {device}")

    stats = ((0.5071, 0.4865, 0.4409), (0.2673, 0.2564, 0.2762))
    t_train = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.AutoAugment(transforms.AutoAugmentPolicy.CIFAR10),
        transforms.ToTensor(),
        transforms.Normalize(*stats),
    ])
    t_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(*stats),
    ])

    root = "./data"
    train_ds = torchvision.datasets.CIFAR100(root=root, train=True, download=True, transform=t_train)
    test_ds = torchvision.datasets.CIFAR100(root=root, train=False, download=True, transform=t_test)

    replay_buffer = DynamicReplayBuffer(REPLAY_BUFFER_CAPACITY)
    accuracy_matrix = np.zeros((N_TASKS, N_TASKS))
    param_counts = []
    results = {
        "accuracy_matrix": accuracy_matrix.tolist(),
        "average_accuracy": [],
        "param_counts": param_counts,
        "tasks": [],
    }

    print("\n🔹 Task 1: Base Initialization (Classes 0-9)")
    train_loader, test_loader = get_task_loaders(train_ds, test_ds, task_id=1)
    model = ResNet18(num_classes=CLASSES_PER_TASK).to(device)
    train_base_model(model, train_loader, device, amp_ok)

    acc = evaluate(model, test_loader, device)
    accuracy_matrix[0, 0] = acc
    print(f"✅ Task 1 Accuracy: {acc:.2f}%")

    replay_buffer.add_exemplars_from_dataset(
        dataset=train_ds,
        classes=get_task_classes(1),
        feature_extractor=model.features,  # 🔴 FIX: Use model.features instead of model
        device=device,
    )

    param_counts.append(sum(p.numel() for p in model.parameters()))
    results["average_accuracy"].append(float(acc))
    results["param_counts"] = param_counts
    results["tasks"].append({
        "task_id": 1,
        "classes": get_task_classes(1),
        "param_count": param_counts[-1],
    })
    results["accuracy_matrix"] = accuracy_matrix.tolist()
    save_progress(results, save_path)

    old_model = copy.deepcopy(model).eval()

    for task_id in range(2, N_TASKS + 1):
        class_start = CLASSES_PER_TASK * (task_id - 1)
        class_end = CLASSES_PER_TASK * task_id - 1
        print(f"\n🔹 Task {task_id}: Incremental Learning (Classes {class_start}-{class_end})")
        current_classes = get_task_classes(task_id)
        train_loader, test_loader = get_task_loaders(train_ds, test_ds, task_id=task_id)

        expand_classifier(model, CLASSES_PER_TASK * task_id)

        replay_loader = replay_buffer.get_loader()
        if replay_loader is None:
            raise RuntimeError("Replay buffer is empty before incremental training.")

        train_incremental_model(model, old_model, train_loader, replay_loader, device, amp_ok)

        replay_buffer.add_exemplars_from_dataset(
            dataset=train_ds,
            classes=current_classes,
            feature_extractor=model.features,  # 🔴 FIX: Use model.features instead of model
            device=device,
        )

        print("  🔸 Evaluation")
        for t in range(1, task_id + 1):
            _, test_loader_t = get_task_loaders(train_ds, test_ds, task_id=t)
            task_acc = evaluate(model, test_loader_t, device)
            accuracy_matrix[task_id - 1, t - 1] = task_acc
            print(f"    Task {t} Accuracy: {task_acc:.2f}%")

        param_counts.append(sum(p.numel() for p in model.parameters()))
        avg_acc = float(np.mean(accuracy_matrix[task_id - 1, :task_id]))
        results["average_accuracy"].append(avg_acc)
        results["accuracy_matrix"] = accuracy_matrix.tolist()
        results["param_counts"] = param_counts
        results["tasks"].append({
            "task_id": task_id,
            "classes": current_classes,
            "param_count": param_counts[-1],
        })
        save_progress(results, save_path)

        old_model = copy.deepcopy(model).eval()

    print("\n🎉 Training Complete!")
    print(f"📊 Final Accuracy Matrix:\n{np.array(results['accuracy_matrix'])}")
    print(f"📈 Average Accuracy: {results['average_accuracy'][-1]:.2f}%")
    print(f"📏 Parameter Count: {results['param_counts'][-1]:,}")
    return results

if __name__ == "__main__":
    train_lwf_10task(seed=0, quick_run=QUICK_RUN)

In [ ]:
'''
ubuntu@gt-ubuntu24-04-cmd-v3-2-120gb-100m:~$ python3 kfn++.py
🚀 Training LwF on Split CIFAR-100 | Seed: 0 | Device: cuda

🔹 Task 1: Base Initialization (Classes 0-9)
✅ Task 1 Accuracy: 92.50%

🔹 Task 2: Incremental Learning (Classes 10-19)
  🔸 Evaluation
    Task 1 Accuracy: 79.90%
    Task 2 Accuracy: 63.60%

🔹 Task 3: Incremental Learning (Classes 20-29)
  🔸 Evaluation
    Task 1 Accuracy: 80.10%
    Task 2 Accuracy: 63.80%
    Task 3 Accuracy: 62.80%

🔹 Task 4: Incremental Learning (Classes 30-39)
  🔸 Evaluation
    Task 1 Accuracy: 81.50%
    Task 2 Accuracy: 71.70%
    Task 3 Accuracy: 66.40%
    Task 4 Accuracy: 46.80%

🔹 Task 5: Incremental Learning (Classes 40-49)
  🔸 Evaluation
    Task 1 Accuracy: 78.60%
    Task 2 Accuracy: 71.00%
    Task 3 Accuracy: 72.20%
    Task 4 Accuracy: 61.70%
    Task 5 Accuracy: 46.30%

🔹 Task 6: Incremental Learning (Classes 50-59)
  🔸 Evaluation
    Task 1 Accuracy: 70.90%
    Task 2 Accuracy: 70.20%
    Task 3 Accuracy: 69.50%
    Task 4 Accuracy: 64.30%
    Task 5 Accuracy: 62.40%
    Task 6 Accuracy: 61.10%

🔹 Task 7: Incremental Learning (Classes 60-69)
  🔸 Evaluation
    Task 1 Accuracy: 68.20%
    Task 2 Accuracy: 69.60%
    Task 3 Accuracy: 71.20%
    Task 4 Accuracy: 66.00%
    Task 5 Accuracy: 66.40%
    Task 6 Accuracy: 67.00%
    Task 7 Accuracy: 50.30%

🔹 Task 8: Incremental Learning (Classes 70-79)
  🔸 Evaluation
    Task 1 Accuracy: 65.70%
    Task 2 Accuracy: 65.60%
    Task 3 Accuracy: 71.00%
    Task 4 Accuracy: 63.30%
    Task 5 Accuracy: 65.60%
    Task 6 Accuracy: 68.00%
    Task 7 Accuracy: 66.40%
    Task 8 Accuracy: 33.80%

    🔹 Task 9: Incremental Learning (Classes 80-89)
  🔸 Evaluation
    Task 1 Accuracy: 66.40%
    Task 2 Accuracy: 65.90%
    Task 3 Accuracy: 70.90%
    Task 4 Accuracy: 62.80%
    Task 5 Accuracy: 64.70%
    Task 6 Accuracy: 70.00%
    Task 7 Accuracy: 66.60%
    Task 8 Accuracy: 53.90%
    Task 9 Accuracy: 47.60%

🔹 Task 10: Incremental Learning (Classes 90-99)
  🔸 Evaluation
    Task 1 Accuracy: 67.00%
    Task 2 Accuracy: 66.40%
    Task 3 Accuracy: 70.70%
    Task 4 Accuracy: 62.60%
    Task 5 Accuracy: 65.00%
    Task 6 Accuracy: 68.40%
    Task 7 Accuracy: 66.70%
    Task 8 Accuracy: 58.80%
    Task 9 Accuracy: 65.90%
    Task 10 Accuracy: 28.30%

🎉 Training Complete!
📊 Final Accuracy Matrix:
[[92.5  0.   0.   0.   0.   0.   0.   0.   0.   0. ]
 [79.9 63.6  0.   0.   0.   0.   0.   0.   0.   0. ]
 [80.1 63.8 62.8  0.   0.   0.   0.   0.   0.   0. ]
 [81.5 71.7 66.4 46.8  0.   0.   0.   0.   0.   0. ]
 [78.6 71.  72.2 61.7 46.3  0.   0.   0.   0.   0. ]
 [70.9 70.2 69.5 64.3 62.4 61.1  0.   0.   0.   0. ]
 [68.2 69.6 71.2 66.  66.4 67.  50.3  0.   0.   0. ]
 [65.7 65.6 71.  63.3 65.6 68.  66.4 33.8  0.   0. ]
 [66.4 65.9 70.9 62.8 64.7 70.  66.6 53.9 47.6  0. ]
 [67.  66.4 70.7 62.6 65.  68.4 66.7 58.8 65.9 28.3]]
📈 Average Accuracy: 61.98%
📏 Parameter Count: 11,220,132
ubuntu@gt-ubuntu24-04-cmd-v3-2-120gb-100m
'''